## Cleaning steps

1. Remove duplicates
2. Remove all row nulls
3. Fill the nulls/''/' '  
4. Trim spaces
5. Standardize case
6. Cast data types
7. Filter invalid rows
7. Rename columns
9. Add metadata columns if applicable
10. Validate schema

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

## Load table

In [0]:
df = spark.table('bikes_catalog.bronze.bronze_sales_details')
df.show()

## Cleaning

In [0]:
df.filter(df.sls_sales != df.sls_price).show()

In [0]:
df = df.dropDuplicates()
df = df.dropna(how='all')

In [0]:
for col in df.columns:
    print(f'Nullls in {col}: {df.filter(F.col(col).isNull()).count()}')

In [0]:
df.filter(F.col('sls_sales').isNull()).show()

In [0]:
df = df.withColumn(
    'sls_sales',
    F.when(
        (F.col('sls_sales').isNull()) | (F.col('sls_sales') <= 0),
        F.abs(F.col('sls_quantity') * F.col('sls_price'))
    ).otherwise(F.col('sls_sales'))
)

In [0]:
df.filter((df.sls_price <= 0) | (df.sls_price.isNull())).show()

In [0]:
df = df.withColumn(
    'sls_price',
    F.when(
        (F.col('sls_price').isNull()) | (F.col('sls_price') <= 0),
        (F.col('sls_sales') / F.col('sls_quantity'))
    ).otherwise(F.col('sls_price'))
)

In [0]:
for col in df.columns:
    print(f'Nullls in {col}: {df.filter(F.col(col).isNull()).count()}')

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name,F.trim(F.col(field.name)))

df.show()

In [0]:
for col in df.columns[3:6]:
    print(f'Invalid value in {col}: {df.filter(F.col(col) == 0).count()}')
    print(f'Invalid length in {col}: {df.filter(F.length(F.col(col)) != 8).count()}',end='\n\n')

In [0]:
df.filter((df.sls_order_dt == 0) | (F.length(df.sls_order_dt) != 8)).show()

In [0]:
df = df.withColumn(
    'sls_order_dt',
    F.when((df.sls_order_dt == 0) | (F.length(df.sls_order_dt) != 8), None)
    .otherwise(df.sls_order_dt)
)

In [0]:
for col in df.columns[3:6]:
    df = df.withColumn(col, F.to_date(col, 'yyyyMMdd'))

df.show()

In [0]:
for old_names, new_names in zip(df.columns, ['Order_number', 'Product_key', 'C_key', 'Order_date', 'Ship_date', 'Due_date', 'Sales', 'Quantity', 'Price']):
    df = df.withColumnRenamed(old_names, new_names)
df.show()

In [0]:
df.printSchema()

In [0]:
df.write.mode('overwrite').saveAsTable('bikes_catalog.silver.silver_sales_details')